## 1. Create the fire calls table

Define an empty Delta table in Unity Catalog (`catalog.schema.table` = `dev.spark_db.sf_fire_calls`) with a fixed column list and types.
The `%sql` magic on the first line makes this cell run as SQL instead of Python.

In [ ]:
%sql
-- Create the table only if it doesn't exist yet, so re-running the cell is safe
CREATE TABLE IF NOT EXISTS dev.spark_db.sf_fire_calls (
  -- Call and incident identifiers
  CallNumber INT, UnitID STRING, IncidentNumber INT, CallType STRING, CallDate DATE,
  -- When the call was logged, how it ended, and when the unit became available
  WatchDate DATE, CallFinalDisposition STRING, AvailableDtTm TIMESTAMP, Address STRING,
  -- Location details (Zipcode is text, not a number, because we never do maths on it)
  City STRING, Zipcode STRING, Battalion STRING, StationArea STRING, Box STRING,
  -- Priority levels and whether an Advanced Life Support unit was sent
  OriginalPriority STRING, Priority STRING, FinalPriority STRING, ALSUnit BOOLEAN,
  -- Call grouping, alarm count, and the type/order of the dispatched unit
  CallTypeGroup STRING, NumAlarms INT, UnitType STRING, UnitSequenceInCallDispatch INT,
  -- City districts and neighbourhood
  FirePreventionDistrict STRING, SupervisorDistrict STRING, Neighborhood STRING,
  -- Coordinates, unique row ID, and response delay in minutes
  Location STRING, RowID STRING, Delay DOUBLE);

## 2. Read the CSV file into a DataFrame

Load the raw San Francisco fire calls CSV from a Unity Catalog Volume.
`inferSchema` lets Spark guess the column types, which is convenient but not always right (see step 5).

In [6]:
# Read the San Francisco fire calls CSV file into a DataFrame
raw_fire_df = (
    # Use the pre-created Spark session; the file is CSV
    spark.read.format("csv")
    # First row has column names, so use them instead of _c0, _c1...
    .option("header", "true")
    # Spark scans the data to guess each column's type (int, string, date...)
    .option("inferSchema", "true")
    # Path to the file in a Unity Catalog Volume
    .load("/Volumes/dev/spark_db/datasets/spark_programming/data/sf-fire-calls.csv")
)


## 3. Count the rows

`count()` is an **action**: it makes Spark actually read the file and return how many rows it holds.

In [4]:
# Count the rows in the DataFrame (this triggers Spark to read the whole file)
raw_fire_df.count()

175296

## 4. Display the data

`display(df)` is a function provided by Databricks Connect that renders the DataFrame as a table.
Note: it's `display(df)`, not `df.display()`. The second form makes Spark look for a column called "display".

In [9]:
# Show the DataFrame as a table (Databricks Connect's display function)
display(raw_fire_df)

,CallNumber,UnitID,IncidentNumber,CallType,CallDate,WatchDate,CallFinalDisposition,AvailableDtTm,Address,City,Zipcode,Battalion,StationArea,Box,OriginalPriority,Priority,FinalPriority,ALSUnit,CallTypeGroup,NumAlarms,UnitType,UnitSequenceInCallDispatch,FirePreventionDistrict,SupervisorDistrict,Neighborhood,Location,RowID,Delay
0,20110016,T13,2003235,Structure Fire,2002-01-11,2002-01-10,Other,01/11/2002 01:51:44 AM,2000 Block of CALIFORNIA ST,SF,94109,B04,38,3362,3,3,3,False,None,1,TRUCK,2,4,5,Pacific Heights,"(37.7895840679362, -122.428071912459)",020110016-T13,2.950000
1,20110022,M17,2003241,Medical Incident,2002-01-11,2002-01-10,Other,01/11/2002 03:01:18 AM,0 Block of SILVERVIEW DR,SF,94124,B10,42,6495,3,3,3,True,None,1,MEDIC,1,10,10,Bayview Hunters Point,"(37.7337623673897, -122.396113802632)",020110022-M17,4.700000
2,20110023,M41,2003242,Medical Incident,2002-01-11,2002-01-10,Other,01/11/2002 02:39:50 AM,MARKET ST/MCALLISTER ST,SF,94102,B03,01,1455,3,3,3,True,None,1,MEDIC,2,3,6,Tenderloin,"(37.7811772186856, -122.411699931232)",020110023-M41,2.433333
3,20110032,E11,2003250,Vehicle Fire,2002-01-11,2002-01-10,Other,01/11/2002 04:16:46 AM,APPLETON AV/MISSION ST,SF,94110,B06,32,5626,3,3,3,False,None,1,ENGINE,1,6,9,Bernal Heights,"(37.7388432849018, -122.423948785199)",020110032-E11,1.500000
4,20110043,B04,2003259,Alarms,2002-01-11,2002-01-10,Other,01/11/2002 06:01:58 AM,1400 Block of SUTTER ST,SF,94109,B04,03,3223,3,3,3,False,None,1,CHIEF,2,4,2,Western Addition,"(37.7872890372638, -122.424236212664)",020110043-B04,3.483333
5,20110072,T08,2003279,Structure Fire,2002-01-11,2002-01-11,Other,01/11/2002 08:03:26 AM,BEALE ST/FOLSOM ST,SF,94105,B03,35,2122,3,3,3,False,None,1,TRUCK,2,3,6,Financial District/South Beach,"(37.7886866619654, -122.392722833778)",020110072-T08,1.750000
6,20110125,E33,2003301,Alarms,2002-01-11,2002-01-11,Other,01/11/2002 09:46:44 AM,0 Block of FARALLONES ST,SF,94112,B09,33,8324,3,3,3,False,None,1,ENGINE,2,9,11,Oceanview/Merced/Ingleside,"(37.7140353531157, -122.454117149916)",020110125-E33,2.716667
7,20110130,E36,2003304,Alarms,2002-01-11,2002-01-11,Other,01/11/2002 09:58:53 AM,600 Block of POLK ST,SF,94102,B02,03,3114,3,3,3,False,None,1,ENGINE,1,2,6,Tenderloin,"(37.7826266328595, -122.41915582123)",020110130-E36,1.783333
8,20110197,E05,2003343,Medical Incident,2002-01-11,2002-01-11,Other,01/11/2002 12:06:57 PM,1500 Block of WEBSTER ST,SF,94115,B04,05,3513,3,3,3,False,None,1,ENGINE,1,4,5,Japantown,"(37.784958590666, -122.431435274503)",020110197-E05,1.516667
9,20110215,E06,2003348,Medical Incident,2002-01-11,2002-01-11,Other,01/11/2002 01:08:40 PM,DIAMOND ST/MARKET ST,SF,94114,B05,06,5415,3,3,3,False,None,1,ENGINE,1,5,8,Castro/Upper Market,"(37.7618954753708, -122.437298717721)",020110215-E06,2.766667


## 5. Check the inferred schema

Print each column's name and the type Spark guessed. Three columns don't match the table from step 1:
- `AvailableDtTm` was read as **string**, but it's a timestamp like `01/11/2002 01:51:44 AM`
- `Zipcode` was read as **integer**, but the table expects a string
- `FinalPriority` was read as **integer**, but the table expects a string

In [10]:
# Print every column with its data type, as inferred by Spark
raw_fire_df.printSchema()

root
 |-- CallNumber: integer (nullable = true)
 |-- UnitID: string (nullable = true)
 |-- IncidentNumber: integer (nullable = true)
 |-- CallType: string (nullable = true)
 |-- CallDate: date (nullable = true)
 |-- WatchDate: date (nullable = true)
 |-- CallFinalDisposition: string (nullable = true)
 |-- AvailableDtTm: string (nullable = true)
 |-- Address: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Zipcode: integer (nullable = true)
 |-- Battalion: string (nullable = true)
 |-- StationArea: string (nullable = true)
 |-- Box: string (nullable = true)
 |-- OriginalPriority: string (nullable = true)
 |-- Priority: string (nullable = true)
 |-- FinalPriority: integer (nullable = true)
 |-- ALSUnit: boolean (nullable = true)
 |-- CallTypeGroup: string (nullable = true)
 |-- NumAlarms: integer (nullable = true)
 |-- UnitType: string (nullable = true)
 |-- UnitSequenceInCallDispatch: integer (nullable = true)
 |-- FirePreventionDistrict: string (nullable = true)
 |-- 

## 6. Fix the column types

`withColumns` takes a dictionary of `{column name: new value}` and returns a **new** DataFrame (`fire_df`).
The original `raw_fire_df` is left unchanged, because DataFrames are immutable.

In [11]:
# to_timestamp turns text into a timestamp; expr lets us write a SQL expression
from pyspark.sql.functions import to_timestamp, expr

# Replace several columns at once and save the result as a new DataFrame
fire_df = raw_fire_df.withColumns({
        # Parse text like "01/11/2002 01:51:44 AM" (hh = 12-hour clock, a = AM/PM)
        "AvailableDtTm": to_timestamp("AvailableDtTm", "MM/dd/yyyy hh:mm:ss a"),
        # Store zip codes as text so they match the table
        "Zipcode": expr("cast(Zipcode as string)"),
        # Store the final priority as text so it matches the table
        "FinalPriority": expr("cast(FinalPriority as string)")
    })

## 7. Save the DataFrame as a table

Write `fire_df` into the Unity Catalog table.
`mode("overwrite")` replaces whatever is already in the table, so re-running this cell won't create duplicates.

In [12]:
# Write the cleaned DataFrame to the table, replacing any existing data
fire_df.write.mode("overwrite").saveAsTable("dev.spark_db.sf_fire_calls")

## 8. Check the saved table with SQL

Query the table to confirm the data was written.
`%sql` on the first line makes the cell run as SQL, and the result is shown as a table.

In [13]:
%sql
-- Count the rows in the saved table; it should match the count from step 3
select count(*) from dev.spark_db.sf_fire_calls

,count(*)
0,175296
